In [87]:
CHAMPION_FEATURES_EXTENDED = [
    "F_raw_agesexcommunes/prop014",
    "F_raw_diplomescommunes/pbac",
    "F_raw_popdepartements/popcom5u",
    "F_raw_capitalimmobiliercommunes/capitalratio",
    "F_raw_cspcommunes/pouem",
    "F_raw_revcommunes/revratiofoy",
    "F_pct_change_agesexcommunes/prop60p",
    "F_pct_change_cspcommunes/paica",
    "F_pct_change_agesexcommunes/pop",
    "F_raw_cspcommunes/pcadr",
    "F_raw_agesexcommunes/prop4059",
    "distancemarseille",
    "F_raw_cspcommunes/paica",
    "F_raw_etrangerscommunes/petranger",
    "F_raw_cspcommunes/pouvr",
    "F_pct_change_cspcommunes/paind",
    "F_pct_change_agesexcommunes/propf",
    "F_pct_change_proprietairescommunes/ppropri",
    "F_raw_cspcommunes/pcapi",
    "F_raw_cspcommunes/pagri",
    "lat",
    "distancelyon",
    "F_pct_change_agesexcommunes/prop014",
    "F_pct_change_agesexcommunes/prop4059",
    "dep_num",
    "F_pct_change_cspcommunes/pcadr",
    "F_pct_change_agesexcommunes/prop1539",
    "F_pct_change_cspcommunes/pouem",
    "F_raw_cspcommunes/ppint",
    "inscrits",
    "F_raw_capitalimmobiliercommunes/prixbien",
    "F_pct_change_capitalimmobiliercommunes/prixbien",
    "F_raw_pibcommunes/pibratio",
    "F_raw_popcommunesvbbm/vbbm",
    "F_raw_agesexcommunes/pop",
    "F_raw_cspcommunes/pindp",
    "F_raw_pibdepartements/pibhab",
    "F_raw_revdepartements/revmoy",
    "F_raw_revimpdepartements/foyimp",
    "F_pct_change_revimpdepartements/foyimp",
    "F_pct_change_revdepartements/revmoy",
    "F_pct_change_etrangerscommunes/petranger",
    "F_pct_change_cspcommunes/pindp",
    "F_raw_cspcommunes/pempl",
    "F_raw_agesexcommunes/age",
    "F_raw_agesexcommunes/prop1539",
    "F_pct_change_cspcommunes/ppint",
    "F_raw_popdepartements/popcoma",
    "F_raw_cspcommunes/paind",
    "F_raw_proprietairescommunes/ppropri",
    "F_raw_diplomescommunes/psup",
    "distanceparis",
    "F_raw_revcommunes/revratio",
    "F_pct_change_cspcommunes/pempl",
    "F_pct_change_cspcommunes/pouvr",
    "F_raw_agesexcommunes/prop60p",
    "F_raw_agesexcommunes/propf",
    "F_pct_change_cspcommunes/pcapi",
    "F_pct_change_agesexcommunes/age",
    "F_pct_change_cspcommunes/pagri",
    "long",
    "F_raw_popcommunesvbbm/vbbmpauvresriches",
    "F_pct_change_revcommunes/revratio",
]

In [88]:
import polars as pl

df = pl.scan_parquet(
    "s3://arthurmanceau/election_modeling_uhcp/data/derived/processed/"
    "data_processed_presidentiel_legislative_from1800_to2027_20260707_143756.parquet/",
    storage_options={
        "aws_endpoint": "https://minio.lab.sspcloud.fr",
        "aws_region": "us-east-1",
        "aws_skip_signature": "true",
    },
).collect()

In [89]:
variable = 'pvoteTD'

In [90]:
X_train = df.filter(pl.col('election_code')=="pres2012").select(CHAMPION_FEATURES_EXTENDED)
X_test = df.filter(pl.col('election_code')=="pres2022").select(CHAMPION_FEATURES_EXTENDED)
X_val = df.filter(pl.col('election_code')=="pres2017").select(CHAMPION_FEATURES_EXTENDED)
y_train = df.filter(pl.col('election_code')=="pres2012").select(variable, 'codecommune')
y_test = df.filter(pl.col('election_code')=="pres2022").select(variable, 'codecommune')
y_val =  df.filter(pl.col('election_code')=="pres2017").select(variable, 'codecommune')

In [91]:
from tabicl import TabICLRegressor

clf = TabICLRegressor(n_estimators=2, device="cpu")  # use n_estimators=8 for best results
clf.fit(X_train.to_pandas(), y_train.select(variable).to_pandas().values.ravel())

,n_estimators,2
,norm_methods,None
,feat_shuffle_method,'latin'
,outlier_threshold,4.0
,batch_size,8
,kv_cache,False
,model_path,None
,allow_auto_download,True
,checkpoint_version,'tabicl-regressor-v2-20260212.ckpt'
,device,'cpu'
,use_amp,'auto'


In [92]:
preds = clf.predict(X_test.to_pandas())

In [93]:
results_tabular = y_test.with_columns(
    pred_tabular=pl.Series(preds)
)

In [94]:
results_boosting = pl.scan_parquet(
    "s3://arthurmanceau/election_modeling_uhcp/data/output/results/results_full_2022_pres_tauTD_tauTG_taupar_7.0.0.parquet",
    storage_options={
        "aws_endpoint": "https://minio.lab.sspcloud.fr",
        "aws_region": "us-east-1",
        "aws_skip_signature": "true",
    },
).collect()
results_previous = pl.scan_parquet(
    "s3://arthurmanceau/election_modeling_uhcp/data/output/results/results_full_2017_pres_tauTD_tauTG_taupar_7.0.0.parquet",
    storage_options={
        "aws_endpoint": "https://minio.lab.sspcloud.fr",
        "aws_region": "us-east-1",
        "aws_skip_signature": "true",
    },
).collect()

In [95]:
results = results_tabular.join(
    results_previous.select(
        'codecommune', f'{variable}_true'
    ).rename(
        {f'{variable}_true':'pred_previous'}
    ), 
    on='codecommune', 
    how='inner'
).join(
    results_boosting.with_columns(
        pl.col(f'{variable}_true').round(3)
    ).select(
        'codecommune', f'{variable}_true', f'{variable}_pred'
    ).rename(
        {f'{variable}_pred': 'pred_boosting', f'{variable}_true':variable}
    ), 
    on='codecommune'
)
assert results.filter((pl.col(variable)-pl.col(f'{variable}_right')).abs()>0.01).height == 0
results = results.drop(f'{variable}_right').with_columns(
    ae_tabular = (pl.col('pred_tabular') - pl.col(variable)).abs(),
    ae_boosting = (pl.col('pred_boosting') - pl.col(variable)).abs(),
    ae_previous = (pl.col('pred_previous') - pl.col(variable)).abs()
)
results.mean()

pvoteTD,codecommune,pred_tabular,pred_previous,pred_boosting,ae_tabular,ae_boosting,ae_previous
f64,str,f32,f64,f64,f64,f64,f64
0.590677,null,0.547743,0.641737,0.639397,0.068132,0.063947,0.060147


FineTuning

In [96]:
import os

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from tabicl import FinetunedTabICLRegressor, TabICLRegressor

In [97]:
is_main_process = int(os.environ.get("LOCAL_RANK", "0")) == 0


def _metrics(pred: np.ndarray, y_true: np.ndarray) -> tuple[float, float, float]:
    return (
        float(mean_squared_error(y_true, pred)),
        float(mean_absolute_error(y_true, pred)),
        float(r2_score(y_true, pred)),
    )

In [98]:
base = TabICLRegressor(n_estimators=2, random_state=0)
base.fit(X_train.to_pandas(), y_train.select(variable).to_pandas().values.ravel())
base_pred = base.predict(X_test.to_pandas())
base_mse, base_mae, base_r2 = _metrics(base_pred, y_test.select(variable).to_pandas().values.ravel())
# Captured for the training-curve reference line in Figure 2.
base_val_mse = float(mean_squared_error(y_val.select(variable).to_pandas().values.ravel(), base.predict(X_val.to_pandas())))

In [99]:
history: dict[str, list[float]] = {
    "epoch": [],
    "val_mse": [],
    "val_mae": [],
    "val_r2": [],
    "train_loss": [],
}


class _HistoryLogger:
    """Record per-epoch validation metrics into ``history``."""

    def setup(self, config):
        del config

    def log_step(self, metrics, step):
        del metrics, step

    def log_epoch(self, metrics, step):
        del step
        history["epoch"].append(int(metrics.get("train/epoch", len(history["epoch"]))) + 1)
        history["val_mse"].append(float(metrics.get("val/mse", np.nan)))
        history["val_mae"].append(float(metrics.get("val/mae", np.nan)))
        history["val_r2"].append(float(metrics.get("val/r2", np.nan)))
        history["train_loss"].append(float(metrics.get("train/mean_loss", np.nan)))

    def finish(self):
        pass


# n_estimators and epochs are kept low for fast doc builds;
# for best results use n_estimators_inference=8, epochs=50+, patience=10.
reg = FinetunedTabICLRegressor(
    epochs=30,
    learning_rate=1e-5,
    n_estimators_finetune=2,
    n_estimators_validation=2,
    n_estimators_inference=2,
    early_stopping=True,
    patience=5,
    random_state=0,
    verbose=True,
)
reg._make_experiment_logger = lambda: _HistoryLogger()
reg.fit(X_train.to_pandas(), y_train.select(variable).to_pandas().values.ravel(), X_val=X_val.to_pandas(), y_val=y_val.select(variable).to_pandas().values.ravel())

/tmp/ipykernel_634055/2953812569.py:45: UserWarning: `output_dir` is not set; no checkpoints will be saved and all fine-tuning progress is lost if the run is interrupted.
  reg.fit(X_train.to_pandas(), y_train.select(variable).to_pandas().values.ravel(), X_val=X_val.to_pandas(), y_val=y_val.select(variable).to_pandas().values.ravel())


Baseline val mse: -0.0120


Fine-tune:   0%|          | 0/30 [00:00<?, ?it/s]

  epoch 1/30:   0%|          | 0/4 [00:00<?, ?it/s]

  epoch 2/30:   0%|          | 0/4 [00:00<?, ?it/s]

  epoch 3/30:   0%|          | 0/4 [00:00<?, ?it/s]

... (14 outputs left out)

  epoch 18/30:   0%|          | 0/4 [00:00<?, ?it/s]

  epoch 19/30:   0%|          | 0/4 [00:00<?, ?it/s]

  epoch 20/30:   0%|          | 0/4 [00:00<?, ?it/s]

,epochs,30
,learning_rate,1e-05
,weight_decay,0.01
,grad_clip,1.0
,amp,True
,use_lr_scheduler,True
,warmup_proportion,0.1
,n_estimators_finetune,2
,n_estimators_validation,2
,n_estimators_inference,2
,max_data_size,10000


In [100]:
ft_pred = reg.predict(X_test.to_pandas())
ft_mse, ft_mae, ft_r2 = _metrics(ft_pred, y_test.select(variable).to_pandas().values.ravel())

if is_main_process:
    header = f"{'metric':<10}{'pretrained':>14}{'fine-tuned':>14}{'delta':>14}"
    rule = "=" * len(header)
    print()
    print(rule)
    print(f"Test-set metrics  (n_train={len(X_train)}, n_test={len(X_test)})")
    print(rule)
    print(header)
    print("-" * len(header))
    print(f"{'MSE':<10}{base_mse:>14.4f}{ft_mse:>14.4f}{ft_mse - base_mse:>+14.4f}")
    print(f"{'MAE':<10}{base_mae:>14.4f}{ft_mae:>14.4f}{ft_mae - base_mae:>+14.4f}")
    print(f"{'R2':<10}{base_r2:>14.4f}{ft_r2:>14.4f}{ft_r2 - base_r2:>+14.4f}")
    print(rule)
    print("lower is better: MSE, MAE;  higher is better: R2")


Test-set metrics  (n_train=36585, n_test=34866)
metric        pretrained    fine-tuned         delta
----------------------------------------------------
MSE               0.0077        0.0059       -0.0018
MAE               0.0695        0.0598       -0.0098
R2                0.0333        0.2618       +0.2285
lower is better: MSE, MAE;  higher is better: R2
